# Notebook 03 — Feature Engineering
**EPPS–American Airlines Data Challenge — GROW 26.2**

Reads cleaned inbound + outbound → joins into sequences → engineers pair-level features → creates binary label `is_risky`

**Output:** `../data/processed/sequences.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

PROC_PATH    = '../data/processed/'
FIGURES_PATH = '../outputs/figures/'
os.makedirs(PROC_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)

print('✅ Imports OK')

✅ Imports OK


## 1 — Load Cleaned Data

In [2]:
inbound  = pd.read_csv(f'{PROC_PATH}inbound_clean.csv',  parse_dates=['Date'])
outbound = pd.read_csv(f'{PROC_PATH}outbound_clean.csv', parse_dates=['Date'])

print(f'Inbound  : {inbound.shape}')
print(f'Outbound : {outbound.shape}')

Inbound  : (276418, 42)
Outbound : (276904, 42)


## 2 — Join Into Sequences

Each sequence = one row representing Airport A → DFW → Airport B on a given date.
We join inbound + outbound on `Date` so DFW weather is shared.

In [ ]:
# ── MEMORY-SAFE JOIN — Quarter by Quarter ──────────────────
import gc
from tqdm import tqdm

PROC_PATH = '../data/processed/'
os.makedirs(PROC_PATH, exist_ok=True)

# Define quarters — adjust months based on your actual data range
QUARTERS = {
    'Q1': [1, 2, 3],
    'Q2': [4, 5, 6],
    'Q3': [7, 8, 9],
    'Q4': [10, 11, 12],
}

quarter_files = []

for q_name, months in QUARTERS.items():

    # Filter inbound + outbound to this quarter only
    in_q  = inbound[inbound['Date'].dt.month.isin(months)].copy()
    out_q = outbound[outbound['Date'].dt.month.isin(months)].copy()

    if len(in_q) == 0 or len(out_q) == 0:
        print(f'{q_name}: no data — skipping')
        continue

    print(f'\n{q_name} — inbound: {len(in_q):,} | outbound: {len(out_q):,}')

    chunks = []
    all_dates = sorted(set(in_q['Date'].unique()) & set(out_q['Date'].unique()))
    print(f'  Common dates: {len(all_dates)}')

    for date in tqdm(all_dates, desc=f'{q_name}'):
        in_day  = in_q[in_q['Date'] == date]
        out_day = out_q[out_q['Date'] == date]

        if len(in_day) == 0 or len(out_day) == 0:
            continue

        # Cross join then filter by valid turnaround window
        in_tmp  = in_day.copy();  in_tmp['_key']  = 1
        out_tmp = out_day.copy(); out_tmp['_key'] = 1

        cross = in_tmp.merge(out_tmp, on='_key', suffixes=('_A', '_B')).drop(columns='_key')

        # Keep only sequences where outbound departs after inbound arrives
        # and turnaround is between 0 and 6 hours
        cross = cross[
            (cross['dep_hour_DFW'] > cross['arr_hour_DFW']) &
            (cross['dep_hour_DFW'] - cross['arr_hour_DFW'] <= 6)
        ]

        if len(cross) > 0:
            chunks.append(cross)

        # Free memory every 30 days
        if len(chunks) % 30 == 0:
            gc.collect()

    if len(chunks) == 0:
        print(f'  {q_name}: no valid sequences found')
        continue

    # Concat this quarter and save to disk immediately
    q_df = pd.concat(chunks, ignore_index=True)
    q_file = f'{PROC_PATH}sequences_{q_name}.csv'
    q_df.to_csv(q_file, index=False)
    quarter_files.append(q_file)

    print(f'  {q_name}: {len(q_df):,} sequences saved → {q_file}')

    # ── Free memory before next quarter ─────────────────────
    del q_df, chunks, in_q, out_q
    gc.collect()

print(f'\n✅ All quarters done. Files saved: {quarter_files}')


Q1 — inbound: 68,141 | outbound: 68,271
  Common dates: 90


Q1:  32%|███▏      | 29/90 [00:28<00:39,  1.54it/s]

In [ ]:
# Save final combined sequences — same file downstream notebooks expect
sequences.to_csv(f'{PROC_PATH}sequences.csv', index=False)
print(f'💾 sequences.csv saved — {sequences.shape}')

In [ ]:
# ── COMBINE ALL QUARTERS ────────────────────────────────────

# Read quarter files one at a time and concat
# Use low_memory=False to avoid dtype warnings on large files
sequences = pd.concat(
    [pd.read_csv(f, parse_dates=['Date']) for f in quarter_files],
    ignore_index=True
)

print(f'Combined sequences shape : {sequences.shape}')
print(f'Unique airport_A         : {sequences["airport_A"].nunique()}')
print(f'Unique airport_B         : {sequences["airport_B"].nunique()}')
print(f'Date range               : {sequences["Date"].min().date()} → {sequences["Date"].max().date()}')
print(f'Quarter breakdown:')
print(sequences.groupby(sequences['Date'].dt.quarter)['Date'].count())

## 3 — Create Target Label: `is_risky`

A sequence is **risky (1)** if EITHER the inbound OR the outbound leg is delayed.
This captures any disruption risk in the A→DFW→B path.

In [ ]:
sequences['is_risky'] = (
    (sequences['delayed_A'] == 1) | (sequences['delayed_B'] == 1)
).astype(int)

print('is_risky distribution:')
print(sequences['is_risky'].value_counts())
print()
print(f'Risky rate: {sequences["is_risky"].mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(7, 4))
counts = sequences['is_risky'].value_counts().reindex([0,1]).fillna(0)
pcts   = sequences['is_risky'].value_counts(normalize=True).reindex([0,1]).fillna(0)*100
bars   = ax.bar(['Not Risky (0)','Risky (1)'], counts.values,
                color=['#2ecc71','#e74c3c'], edgecolor='white', width=0.5)
for bar, cnt, pct in zip(bars, counts.values, pcts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(counts)*0.01,
            f'{int(cnt):,}\n({pct:.1f}%)', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Sequence Risk Label Distribution', fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, max(counts)*1.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}13_sequence_label_dist.png', bbox_inches='tight')
plt.show()
print('💾 13_sequence_label_dist.png')

## 4 — Engineer Pair-Level Features

In [ ]:
# ── Weather combination features ────────────────────────────
# Both endpoints stormy at the same time = worst case
sequences['both_thunderstorm'] = (
    (sequences['origin_thunderstorm'] == 1) &
    (sequences['dest_thunderstorm']   == 1)
).astype(int)

sequences['both_low_visibility'] = (
    (sequences['origin_visibility_mi'] < 3) &
    (sequences['dest_visibility_mi']   < 3)
).astype(int)

sequences['both_freezing'] = (
    (sequences['origin_freezing'] == 1) &
    (sequences['dest_freezing']   == 1)
).astype(int)

sequences['both_low_ceiling'] = (
    (sequences['origin_low_ceiling'] == 1) &
    (sequences['dest_low_ceiling']   == 1)
).astype(int)

# Total weather risk score across both airports + DFW
# Use _A suffix for DFW cols (they're identical; just pick one side)
sequences['total_wx_risk'] = (
    sequences['origin_thunderstorm'] +
    sequences['dest_thunderstorm']   +
    sequences['origin_precipitation'] +
    sequences['dest_precipitation']  +
    sequences['origin_freezing']     +
    sequences['dest_freezing']       +
    sequences['origin_low_ceiling']  +
    sequences['dest_low_ceiling']
)

# DFW weather as amplifier
dfw_wx_col = 'dfw_thunderstorm_A' if 'dfw_thunderstorm_A' in sequences.columns else 'dfw_thunderstorm'
dfw_pr_col = 'dfw_precipitation_A' if 'dfw_precipitation_A' in sequences.columns else 'dfw_precipitation'
dfw_lc_col = 'dfw_low_ceiling_A'   if 'dfw_low_ceiling_A'   in sequences.columns else 'dfw_low_ceiling'
dfw_fr_col = 'dfw_freezing_A'      if 'dfw_freezing_A'      in sequences.columns else 'dfw_freezing'

sequences['dfw_wx_risk'] = (
    sequences[dfw_wx_col] +
    sequences[dfw_pr_col] +
    sequences[dfw_lc_col] +
    sequences[dfw_fr_col]
)

# Absolute difference in visibility (signal for asymmetric risk)
sequences['visibility_diff'] = abs(
    sequences['origin_visibility_mi'] - sequences['dest_visibility_mi']
)
sequences['wind_diff'] = abs(
    sequences['origin_wind_kts'] - sequences['dest_wind_kts']
)

print('✅ Weather features created')

In [ ]:
# ── Operational combination features ────────────────────────
# Combined delay signal across both legs
sequences['total_block_delay']    = sequences['block_delay_A']    + sequences['block_delay_B']
sequences['total_airborne_delay'] = sequences['airborne_delay_A'] + sequences['airborne_delay_B']
sequences['total_taxi_out']       = sequences['taxi_out_delay_A'] + sequences['taxi_out_delay_B']
sequences['total_taxi_in']        = sequences['taxi_in_delay_A']  + sequences['taxi_in_delay_B']

# On-time rate combination
sequences['avg_ontime_pct'] = (
    sequences['pct_ontime_gate_arr_A'] + sequences['pct_ontime_gate_dep_B']
) / 2

sequences['min_ontime_pct'] = sequences[[
    'pct_ontime_gate_arr_A', 'pct_ontime_gate_dep_B'
]].min(axis=1)

# Total DFW congestion (EDCT = ground delay programs)
edct_A = 'edct_count_A' if 'edct_count_A' in sequences.columns else 'edct_count'
edct_B = 'edct_count_B' if 'edct_count_B' in sequences.columns else 'edct_count'
sequences['total_edct'] = sequences[edct_A] + sequences[edct_B]

# Departure hour gap (how tight is the connection at DFW)
sequences['dfw_connection_gap'] = sequences['dep_hour_DFW'] - sequences['arr_hour_DFW']

print('✅ Operational features created')

## 5 — Feature Summary

In [ ]:
new_features = [
    'both_thunderstorm','both_low_visibility','both_freezing','both_low_ceiling',
    'total_wx_risk','dfw_wx_risk','visibility_diff','wind_diff',
    'total_block_delay','total_airborne_delay','total_taxi_out','total_taxi_in',
    'avg_ontime_pct','min_ontime_pct','total_edct','dfw_connection_gap'
]

print('=== NEW ENGINEERED FEATURES ===')
for f in new_features:
    if f in sequences.columns:
        print(f'  {f:30s} | mean={sequences[f].mean():.2f} | null={sequences[f].isnull().sum()}')

print(f'\nTotal columns in sequences: {sequences.shape[1]}')

## 6 — Engineered Feature vs Risk

In [ ]:
plot_feats = ['total_wx_risk','dfw_wx_risk','total_block_delay',
              'avg_ontime_pct','dfw_connection_gap','visibility_diff']
avail_pf = [f for f in plot_feats if f in sequences.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(avail_pf):
    ax = axes[i]
    sequences[sequences['is_risky']==0][col].hist(
        ax=ax, bins=20, alpha=0.65, color='#2ecc71',
        label='Not Risky', density=True)
    sequences[sequences['is_risky']==1][col].hist(
        ax=ax, bins=20, alpha=0.65, color='#e74c3c',
        label='Risky', density=True)
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)

for j in range(len(avail_pf), len(axes)): axes[j].set_visible(False)
plt.suptitle('Engineered Features — Risky vs Not Risky Sequences',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}14_engineered_features.png', bbox_inches='tight')
plt.show()
print('💾 14_engineered_features.png')

## 7 — Risk Rate by Season & Airport Pair

In [ ]:
# Use season_A (inbound season)
season_col = 'season_A' if 'season_A' in sequences.columns else 'season'
season_risk = sequences.groupby(season_col)['is_risky'].mean() * 100
season_order = ['winter','spring','summer','fall']
season_risk = season_risk.reindex([s for s in season_order if s in season_risk.index])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Season
ax = axes[0]
cols_s = ['#3498db','#2ecc71','#e74c3c','#f39c12'][:len(season_risk)]
bars = ax.bar(season_risk.index, season_risk.values, color=cols_s, edgecolor='white', width=0.5)
for bar, val in zip(bars, season_risk.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Sequence Risk Rate by Season', fontweight='bold')
ax.set_ylabel('Risk Rate (%)')
ax.set_ylim(0, 100)
ax.axhline(sequences['is_risky'].mean()*100, color='k', ls='--', alpha=0.4, label='Overall')
ax.legend()

# Top risky pairs
top_pairs = (sequences.groupby(['airport_A','airport_B'])['is_risky']
             .mean() * 100
             .sort_values(ascending=False)
             .head(15)
             .reset_index())
top_pairs['pair'] = top_pairs['airport_A'] + ' → ' + top_pairs['airport_B']
ax2 = axes[1]
colors_p = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(top_pairs)))
ax2.barh(top_pairs['pair'], top_pairs['is_risky'], color=colors_p, edgecolor='white')
ax2.set_title('Top 15 Riskiest Airport Pairs', fontweight='bold')
ax2.set_xlabel('Risk Rate (%)')
ax2.invert_yaxis()

plt.suptitle('Sequence Risk Patterns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}15_risk_by_season_pairs.png', bbox_inches='tight')
plt.show()
print('💾 15_risk_by_season_pairs.png')

## 8 — Save Sequences

In [ ]:
sequences.to_csv(f'{PROC_PATH}sequences.csv', index=False)
print(f'💾 Saved → data/processed/sequences.csv')
print(f'   Shape  : {sequences.shape}')
print(f'   Risky  : {sequences["is_risky"].sum():,} ({sequences["is_risky"].mean()*100:.1f}%)')
print(f'   Not Risky: {(sequences["is_risky"]==0).sum():,}')
print()
print('✅ Next → Notebook 04: Modeling')